# Integrated Sentiment Analysis Pipeline (Robust API-Based Version)
This notebook integrates the entire process of crawling products, preprocessing data, predicting sentiments using the trained GRU model, and generating visual reports.

In [ ]:
import os
import re
import time
import pickle
import urllib.request
from urllib.request import urlretrieve
from collections import Counter

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, Dense, GRU
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from eunjeon import Mecab

In [ ]:
stopwords = ['도', '는', '다', '의', '가', '이', '은', '한', '에', '하', '고', '을', '를', '인', '듯', '과', '와', '네', '들', '듯', '지', '임', '게']
max_len = 80

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r"[^ㄱ-ㅎㅏ-ㅣ가-힣 ]", "", text).strip()

In [ ]:
# Model & Tokenizer loading or training
model_path = 'best_model.h5'
tokenizer_path = 'tokenizer.pickle'

if os.path.exists(model_path) and os.path.exists(tokenizer_path):
    print("Loading existing model and tokenizer...")
    loaded_model = load_model(model_path)
    with open(tokenizer_path, 'rb') as handle:
        tokenizer = pickle.load(handle)
else:
    print("Model/Tokenizer not found. Training a new model from scratch...")
    urlretrieve("https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt", filename="ratings_total.txt")
    total_data = pd.read_table('ratings_total.txt', names=['ratings', 'reviews'])
    total_data['label'] = np.select([total_data.ratings > 3], [1], default=0)
    total_data.drop_duplicates(subset=['reviews'], inplace=True)
    
    train_data, test_data = train_test_split(total_data, test_size=0.25, random_state=42)
    train_data['reviews'] = train_data['reviews'].apply(clean_text)
    train_data.dropna(subset=['reviews'], inplace=True)
    
    mecab = Mecab()
    train_data['tokenized'] = train_data['reviews'].apply(mecab.morphs).apply(lambda x: [w for w in x if w not in stopwords])
    
    X_train = train_data['tokenized'].values
    y_train = train_data['label'].values
    
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(X_train)
    
    threshold = 2
    total_cnt = len(tokenizer.word_index)
    rare_cnt = sum(1 for v in tokenizer.word_counts.values() if v < threshold)
    vocab_size = total_cnt - rare_cnt + 2
    
    tokenizer = Tokenizer(vocab_size, oov_token='OOV')
    tokenizer.fit_on_texts(X_train)
    X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len)
    
    model = Sequential([
        Embedding(vocab_size, 100),
        GRU(128),
        Dense(1, activation='sigmoid')
    ])
    
    es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=3)
    mc = ModelCheckpoint(model_path, monitor='val_acc', mode='max', verbose=1, save_best_only=True)
    model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['acc'])
    model.fit(X_train_seq, y_train, epochs=5, callbacks=[es, mc], batch_size=128, validation_split=0.2)
    
    loaded_model = load_model(model_path)
    with open(tokenizer_path, 'wb') as handle:
        pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
    print("Training finished and model saved.")

In [ ]:
# Crawler section - Search setup
search = input('검색어를 입력하세요: ')
url = "https://www.musinsa.com/"

options = webdriver.ChromeOptions()
browser = webdriver.Chrome(options=options)
browser.get(url)

if not os.path.exists(search):
    os.mkdir(search)

In [ ]:
# Perform search & extract using broad URL pattern matching
search_box = browser.find_element(By.XPATH, '//*[@class="search head-search-inp keyword-dec"]')
search_box.send_keys(search)
time.sleep(1)

full_html = browser.page_source
soup = BeautifulSoup(full_html, 'html.parser')

a_tags = soup.find_all('a', href=re.compile(r'/goods/\d+'))

brand_recommend = []
item_recommend = []
goods_ids = []
seen_ids = set()

for a in a_tags:
    href = a.get('href', '')
    match = re.search(r'/goods/(\d+)', href)
    if match:
        gid = match.group(1)
        if gid not in seen_ids:
            text = a.text.strip()
            if len(text) > 5:
                lines = [line.strip() for line in text.split('\n') if line.strip()]
                brand = lines[0] if len(lines) > 0 else "Brand"
                name = lines[1] if len(lines) > 1 else lines[0]
                
                brand_recommend.append(brand)
                item_recommend.append(name)
                goods_ids.append(gid)
                seen_ids.add(gid)

recommend_df = pd.DataFrame({'브랜드': brand_recommend, '상품명': item_recommend})
print(recommend_df)

search_num = int(input('원하는 상품의 번호를 입력하세요: '))
selected_goods_id = goods_ids[search_num]
selected_goods_name = item_recommend[search_num]
selected_brand = brand_recommend[search_num]

browser.get(f"https://www.musinsa.com/goods/{selected_goods_id}")
time.sleep(3)

In [ ]:
# Extract Product Details (Title & Representative Image) using OpenGraph Meta Tags
full_html = browser.page_source
soup = BeautifulSoup(full_html, 'html.parser')

meta_image = soup.find('meta', property='og:image')
image_src = meta_image['content'] if meta_image else ""

meta_title = soup.find('meta', property='og:title')
if meta_title:
    result_name = meta_title['content'].strip().split(' - ')[0].strip()
else:
    result_name = selected_goods_name

result_name = re.sub(r'[\/:*?"<>|]', '_', result_name)

if image_src:
    if image_src.startswith('//'):
        image_src = 'https:' + image_src
    urlretrieve(image_src, f'{search}/{result_name}.jpg')

# Close Browser - Selenium no longer needed
browser.quit()

In [ ]:
# Robust API Crawling Function
def crawl_reviews_api(goods_no, total_to_collect, has_photo="false"):
    url = "https://www.musinsa.com/api2/review/v1/view/list"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36",
        "Referer": f"https://www.musinsa.com/goods/{goods_no}"
    }
    
    score_list = []
    text_list = []
    page = 1
    count = 0
    
    while count < total_to_collect:
        params = {
            "goodsNo": goods_no,
            "page": page,
            "pageSize": 20,
            "sort": "up_cnt_desc",
            "myFilter": "false",
            "hasPhoto": has_photo
        }
        
        try:
            response = requests.get(url, params=params, headers=headers)
            if response.status_code != 200:
                break
                
            res_json = response.json()
            data = res_json.get("data", {})
            review_list = data.get("list", [])
            
            if not review_list:
                break
                
            for rev in review_list:
                content = rev.get("content", "").strip()
                score = rev.get("score", None)
                
                if not content:
                    content = rev.get("reviewContent", "").strip()
                if score is None:
                    score = rev.get("point", 0)
                    
                if content:
                    text_list.append(content)
                    score_list.append(score)
                    count += 1
                    
                if count >= total_to_collect:
                    break
                    
            page += 1
            time.sleep(0.5)
        except Exception:
            break
            
    return score_list, text_list

In [ ]:
limit = int(input("수집할 리뷰 개수를 입력하세요 (예: 100): "))

print("Starting Crawling process (API)...")
general_scores, general_texts = crawl_reviews_api(selected_goods_id, limit, has_photo="false")
photo_scores, photo_texts = crawl_reviews_api(selected_goods_id, limit, has_photo="true")

In [ ]:
# Create & Save DataFrames
def create_and_save_df(scores, texts, suffix):
    df = pd.DataFrame({'별점': scores, '리뷰내용': texts})
    df['리뷰내용'] = df['리뷰내용'].str.replace("\t", "", regex=False).str.replace("\n", "", regex=False)
    df.to_csv(f"{search}/{result_name}_{suffix}.csv", encoding="utf-8-sig", index=True)
    return df

df_general = create_and_save_df(general_scores, general_texts, 'general')
df_photo = create_and_save_df(photo_scores, photo_texts, 'photo')

df_final = pd.DataFrame({
    '별점': general_scores + photo_scores,
    '리뷰내용': general_texts + photo_texts
})
df_final['리뷰내용'] = df_final['리뷰내용'].str.replace("\t", "", regex=False).str.replace("\n", "", regex=False)
df_final.to_csv(f"{search}/{result_name}_final.csv", encoding="utf-8-sig", index=True)
print("Crawled data saved successfully!")

In [ ]:
# Sentiment analysis execution
mecab = Mecab()
positive_ave = []
negative_ave = []
positive_data = []
negative_data = []

def predict_sentiment(new_sentence):
    sentence_nouns = mecab.nouns(new_sentence)
    cleaned = clean_text(new_sentence)
    tokens = mecab.morphs(cleaned)
    tokens = [word for word in tokens if word not in stopwords]
    
    if not tokens:
        return
        
    encoded = tokenizer.texts_to_sequences([tokens])
    padded = pad_sequences(encoded, maxlen=max_len)
    score = float(loaded_model.predict(padded, verbose=0)[0][0])
    
    if score > 0.5:
        positive_ave.append(score * 100)
        positive_data.append(sentence_nouns)
    else:
        negative_ave.append((1 - score) * 100)
        negative_data.append(sentence_nouns)

print("Predicting review sentiment...")
reviews = df_final['리뷰내용'].dropna().tolist()
for r in reviews:
    predict_sentiment(r)
print(f"Finished: {len(positive_ave)} positive, {len(negative_ave)} negative")

In [ ]:
# Pie chart for Sentiment Ratio
total_reviews = len(positive_ave) + len(negative_ave)
if total_reviews > 0:
    ratio_list = [(len(positive_ave) / total_reviews) * 100, (len(negative_ave) / total_reviews) * 100]
    
    # Set Matplotlib font to support Korean characters
    try:
        from matplotlib import font_manager, rc
        if os.name == 'nt':
            font_path = "C:/Windows/Fonts/malgun.ttf"
            font_name = font_manager.FontProperties(fname=font_path).get_name()
            rc('font', family=font_name)
        else:
            rc('font', family='AppleGothic')
    except Exception as e:
        print("Matplotlib Korean font setup failed:", e)
        
    plt.figure(figsize=(6, 6))
    plt.pie(ratio_list, labels=['Positive', 'Negative'], autopct='%.2f%%', colors=['#4CAF50', '#F44336'])
    plt.legend(['Positive', 'Negative'])
    plt.title(f'<{result_name}> Review Sentiment Ratio')
    plt.show()
else:
    print("No reviews found to visualize.")

In [ ]:
# Bar charts for top keywords
# Automatically extract keywords except brand name
exception_list = [selected_brand, selected_goods_name]
exception_list += mecab.nouns(selected_brand) + mecab.nouns(selected_goods_name)
exception_list = list(set(exception_list))

def plot_top_keywords(noun_data, title, color):
    all_nouns = [noun for nouns in noun_data for noun in nouns]
    filtered_nouns = [n for n in all_nouns if len(n) > 1 and n not in exception_list and n not in stopwords]
    
    counter = Counter(filtered_nouns)
    top_keywords = counter.most_common(15)
    
    if not top_keywords:
        print(f"Insufficient keywords for {title}")
        return
        
    words = [item[0] for item in top_keywords]
    counts = [item[1] for item in top_keywords]
    
    plt.figure(figsize=(10, 5))
    plt.bar(words, counts, color=color)
    plt.title(f'{title} - Top Keywords')
    plt.xticks(rotation=45)
    plt.ylabel('Counts')
    plt.tight_layout()
    plt.show()

print("Positive Reviews Keywords:")
plot_top_keywords(positive_data, 'Positive Reviews', '#4CAF50')

print("\nNegative Reviews Keywords:")
plot_top_keywords(negative_data, 'Negative Reviews', '#F44336')